In [ ]:
import base64
import time
import pandas as pd
import os
import google.generativeai as genai
from IPython.display import display
from IPython.display import Markdown

# Gemini API Key - Set this as an environment variable or replace with your actual key
os.environ["GEMINI_API_KEY"] = ""

# Configure the Gemini API
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# Read CSV
questions_df = pd.read_csv("C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs _Bad Prompt/VLAT Questions.csv")
questions = questions_df.to_dict('records')
responses_data = []

for i, question in enumerate(questions, start=1):
    time.sleep(5)
    try:
        print(f"\nProcessing question {i}:")
        print(question)
        
        # Path to your image
        image_path = "C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/" + str(question.get('vis', '')) + ".png"
        
        # Question text
        question_text = question.get('question: ', '')
        question_options = question.get('option:', '')
        correct_ans = str(question.get('correct', '')).strip()
        
        print(f"Processing image: {image_path}")
        print(f"Question: {question_text}")
        print(f"Options: {question_options}")
        print(f"Correct answer: {correct_ans}")
        
        # Read the image as binary and create Gemini image part
        with open(image_path, "rb") as image_file:
            image_data = image_file.read()
        
        # Define the prompt
        prompt = ('I am about to show you an image and ask you a multiple choice question about that image. Select the BEST answer, based only on the chart and not external knowledge.' +
                'End with: "Correct Answer: ". Write the value in "Correct Answer: " what you got from "API Response". Just write the value, nothing else. Do not write anything after this. \n\n' +
                 '\n\n' + question_text + " " + question_options)
        
        # Create Gemini content parts - removed as we're using the new API format
        
        # Add retry logic for any API errors
        max_retries = 3
        retry_delay = 20  # seconds
        
        for retry in range(max_retries):
            try:
                # Use the Gemini 2.0 Pro model
                model = genai.GenerativeModel('gemini-2.0-pro-exp-02-05')
                
                # Create parts list with text and image
                parts = [
                    {"text": prompt},
                    {"inline_data": {
                        "mime_type": "image/png",
                        "data": base64.b64encode(image_data).decode('utf-8')
                    }}
                ]
                
                # Configure generation parameters
                generation_config = {
                    "temperature": 0.0,  # Match your GPT temperature of 0.0
                    "max_output_tokens": 5000,
                }
                
                time_start = time.perf_counter()
                
                # Generate the response
                response = model.generate_content(
                    parts,
                    generation_config=generation_config
                )
                
                time_end = time.perf_counter()
                
                # If we get here, response was successful
                break
                
            except Exception as e:
                print(f"Error during API call (attempt {retry + 1}/{max_retries}):", str(e))
                if retry < max_retries - 1:
                    print(f"\nAPI error, waiting {retry_delay} seconds before retry {retry + 1}/{max_retries}")
                    time.sleep(retry_delay)
                    continue
                raise  # Re-raise the last exception if we've exhausted all retries
        
        try:
            # Extract the full response text
            full_response = response.text
            print("\nAPI Response:", full_response[:200] + "...")  # Debug print - first 200 chars
            
            # Extract the answer after "Correct Answer: "
            if "Correct Answer: " in full_response:
                gpt_answer = full_response.split("Correct Answer: ")[-1].strip()
                # Case-insensitive comparison after stripping whitespace
                is_correct = gpt_answer.strip().upper() == correct_ans.strip().upper()
            else:
                gpt_answer = "Error: No answer in correct format"
                is_correct = "N/A"
                
        except Exception as e:
            print(f"Error processing response: {str(e)}")
            gpt_answer = f"Error: {str(e)}"
            is_correct = "N/A"
        
        responses_data.append([gpt_answer, time_end-time_start, is_correct])
        print(f"\nAnswer: {gpt_answer}")
        print(f"Time taken: {time_end-time_start:.2f} seconds")
        print(f"Correct? {is_correct}")
        
        # Increase delay between requests to 15 seconds
        time.sleep(max(15 - (time_end-time_start), 0))
        
    except Exception as e:
        print(f"Error processing question {i}:", str(e))
        responses_data.append([f"Error: {str(e)}", 0, "N/A"])

# Create Results directory if it doesn't exist
results_dir = "C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs _Bad Prompt/Results/3_GEMNI 2.0 Pro_Final Results with Explanation/"
os.makedirs(results_dir, exist_ok=True)

# Save results
results_df = pd.DataFrame(responses_data, columns=['response', 'time', 'correct_bool'])
results_df.index = range(1, results_df.shape[0] + 1)
results_df.to_csv(results_dir + "Gemini_VLAT_" + str(int(time.time())) + ".csv", index_label="id")
print("\n*** Finished ***")


Processing question 1:
{'id': 1, 'dropped': 'no', 'vis': 'VLAT_a', 'item': 'a_1', 'question: ': 'What was the price of a barrel of oil in February 2015? ', 'option:': '$39.72; $57.36; $48.90; $62.85', 'correct': '$48.90 '}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_a.png
Question: What was the price of a barrel of oil in February 2015? 
Options: $39.72; $57.36; $48.90; $62.85
Correct answer: $48.90

API Response: Correct Answer: $48.90...

Answer: $48.90
Time taken: 1.30 seconds
Correct? True

Processing question 2:
{'id': 2, 'dropped': 'no', 'vis': 'VLAT_a', 'item': 'a_2', 'question: ': 'In which month was the price of a barrel of oil the lowest in 2015?', 'option:': 'April; June; September; December', 'correct': 'April'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_a.png
Question: In which month was the price of a barrel of oil the lowest in 2015?
Options: April; June; September; Decemb


API Response: True
Correct Answer: 
...

Answer: 
Time taken: 1.01 seconds
Correct? False

Processing question 16:
{'id': 16, 'dropped': 'no', 'vis': 'VLAT_d', 'item': 'd_1', 'question: ': 'What is the approval rating of Republicans among the people who have the education level of Postgraduate Study?', 'option:': '38%; 47%; 53%; 75%', 'correct': '75%'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_d.png
Question: What is the approval rating of Republicans among the people who have the education level of Postgraduate Study?
Options: 38%; 47%; 53%; 75%
Correct answer: 75%

API Response: Correct Answer: 75%
...

Answer: 75%
Time taken: 0.91 seconds
Correct? True

Processing question 17:
{'id': 17, 'dropped': 'no', 'vis': 'VLAT_d', 'item': 'd_2', 'question: ': 'What is the education level of people in which the Democrats have the lowest approval rating?', 'option:': 'High School Graduate or Less; Some College Degree; College Graduate; Postgra


API Response: Correct Answer: 53.9 - 116.3 kg
...

Answer: 53.9 - 116.3 kg
Time taken: 1.17 seconds
Correct? True

Processing question 30:
{'id': 30, 'dropped': 'yes', 'vis': 'VLAT_g', 'item': 'g_4', 'question: ': 'More than 60 percent of males lie between the weights of 60 kg and 100 kg.', 'option:': 'True; False', 'correct': 'TRUE'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_g.png
Question: More than 60 percent of males lie between the weights of 60 kg and 100 kg.
Options: True; False
Correct answer: TRUE

API Response: TrueCorrect Answer: 
...

Answer: 
Time taken: 0.92 seconds
Correct? False

Processing question 31:
{'id': 31, 'dropped': 'no', 'vis': 'VLAT_g', 'item': 'g_5', 'question: ': 'No person weighs more than 120 kg.', 'option:': 'True; False', 'correct': 'TRUE'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_g.png
Question: No person weighs more than 120 kg.
Options: True; False


API Response: Correct Answer: 500 - 4,700
...

Answer: 500 - 4,700
Time taken: 8.55 seconds
Correct? False

Processing question 44:
{'id': 44, 'dropped': 'no', 'vis': 'VLAT_j', 'item': 'j_5', 'question: ': 'The number of girls named ‘Isla’ was __________ from 2009 to 2012.', 'option:': 'rising; falling; staying', 'correct': 'rising'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_j.png
Question: The number of girls named ‘Isla’ was __________ from 2009 to 2012.
Options: rising; falling; staying
Correct answer: rising

API Response: Correct Answer: rising
...

Answer: rising
Time taken: 0.93 seconds
Correct? True

Processing question 45:
{'id': 45, 'dropped': 'no', 'vis': 'VLAT_j', 'item': 'j_6', 'question: ': 'In the UK, the number of girls named ‘Amelia’ in 2014 was more than it was in 2013.', 'option:': 'True; False', 'correct': 'TRUE'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_j.png
Que


API Response: Correct Answer: True
...

Answer: True
Time taken: 1.00 seconds
Correct? False

Processing question 58:
{'id': 58, 'dropped': 'yes', 'vis': 'VLAT_l', 'item': 'l_1', 'question: ': 'Out of the total number of unique visitors across all the websites, about what percentage of unique visitors were from Google in 2010?', 'option:': '1%; 7%; 20%; 30%', 'correct': '7%'}
Processing image: C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Images/VLAT_l.png
Question: Out of the total number of unique visitors across all the websites, about what percentage of unique visitors were from Google in 2010?
Options: 1%; 7%; 20%; 30%
Correct answer: 7%

API Response: Correct Answer: 20%
...

Answer: 20%
Time taken: 0.95 seconds
Correct? False

Processing question 59:
{'id': 59, 'dropped': 'no', 'vis': 'VLAT_l', 'item': 'l_2', 'question: ': 'For which website was the number of unique visitors the largest in 2010?', 'option:': 'Facebook; Amazon; Bing; Google', 'correct': 'Google'